# Testing DATA integration through Tango

This notebook checks the real-hardware DATA flow:

1. Connect to Tango devices.
2. Configure the DATA Tango device.
3. Acquire a STEM image from ThermoMicroscope.
4. Use the returned saved file path to retrieve data through the DATA device.


### Quick Start Code Cell

In [ ]:
import subprocess
import os
import json
import time

import tango
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Tango / DATA configuration
DB_HOST = "localhost"
DB_PORT = 9000

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

SCAN_DEVICE = "asyncroscopy/scan/default"
MICROSCOPE_DEVICE = "asyncroscopy/microscope/default"
DATA_DEVICE = "asyncroscopy/data/default"

TILED_HOST = "10.46.217.241"
TILED_PORT = 9091
SAVE_PATH = "D:/microscopedata/ahoust17/20260511/"  # Change this to the directory watched by Tiled

print("TANGO_HOST =", os.environ["TANGO_HOST"])


## 0. Start local Tango servers


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DB_HOST = "localhost"
DB_PORT = 9000
server_list = [("stage", "asyncroscopy.hardware.STAGE"),
               ("scan", "asyncroscopy.hardware.SCAN"),
               ("eds", "asyncroscopy.detectors.EDS"),
               ("camera", "asyncroscopy.detectors.CAMERA"),
               ("data", "asyncroscopy.software.DATA")]
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = os.path.dirname(os.getcwd())
os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"
env = {**os.environ, "TANGO_HOST": f"{DB_HOST}:{DB_PORT}"}
processes = {}

def popen(cmd):
    return subprocess.Popen(cmd, env=env, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def wait_for_device(name, timeout=60, interval=1):
    print(f"  Waiting for {name}...", end="", flush=True)
    start = time.time()
    while time.time() - start < timeout:
        try:
            tango.DeviceProxy(name).ping()
            print(f" ✅ ready ({time.time()-start:.1f}s)")
            return True
        except Exception:
            print(".", end="", flush=True)
            time.sleep(interval)
    print(f" ❌ timed out after {timeout}s")
    return False

def check_processes(*names):
    for name in names:
        p = processes[name]
        print(f"\n─── {name} (PID {p.pid}) ───\n  Running: {p.poll() is None}")
        for label, fd in [("STDOUT", p.stdout), ("STDERR", p.stderr)]:
            try:
                print(f"  {label}: {fd.read1(4096).decode() or '(empty)'}")
            except Exception:
                print(f"  {label}: (no output yet)")

# Kill old processes (if any)
print("Clearing old processes...")
for cmd in [f"kill -9 $(lsof -t -i:{DB_PORT}) 2>/dev/null || true",
            *[f"pkill -f '{module.split('.')[-1]} {name}_instance' 2>/dev/null || true"
              for name, module in server_list],
            "pkill -f 'ThermoMicroscope microscope_instance' 2>/dev/null || true"]:
    subprocess.run(cmd, shell=True)
time.sleep(2)

# Start DB
print(f"Project dir: {PROJECT_DIR}\nStarting Tango Database...")
processes["database"] = popen(["uv", "run", "python", "-m", "tango.databaseds.database", "2"])
print("  Waiting for database...", end="", flush=True)
for _ in range(30):
    try:
        tango.Database(DB_HOST, DB_PORT)
        print(" ✅ ready")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(1)
else:
    check_processes("database")
    raise RuntimeError("Tango database failed to start.")

# Register devices
print("Registering devices...")
r = subprocess.run(["uv", "run", "scripts/2_register_devices.py"], env=env, cwd=PROJECT_DIR, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print("ERROR:", r.stderr)
    raise RuntimeError("Device registration failed — stopping here.")

# Start servers
print("Starting device servers...")
for name, module in server_list:
    processes[name] = popen(["uv", "run", "python", "-m", module, f"{name}_instance"])

if not all(wait_for_device(f"asyncroscopy/{d}/default") for d in ["stage", "scan", "eds", "camera", "data"]):
    print("\n⚠️  Debug info:")
    check_processes("stage", "scan", "eds", "camera", "data")
    raise RuntimeError("One or more device servers failed.")

print("Starting Microscope...")
processes["microscope"] = popen(["uv", "run", "python", "-m", "asyncroscopy.ThermoMicroscope", "microscope_instance"])
if not wait_for_device("asyncroscopy/microscope/default"):
    print("\n⚠️  Debug info:")
    check_processes("microscope")
    raise RuntimeError("Microscope server failed — see debug info above.")

print("\n✅ All servers ready!")


## 1. Connect to devices

In [ ]:
db = tango.Database()
print("Devices registered in Tango DB:\n")
for device in db.get_device_name("*", "*"):
    print(device)


In [ ]:
scan = tango.DeviceProxy(SCAN_DEVICE)
microscope = tango.DeviceProxy(MICROSCOPE_DEVICE)
data = tango.DeviceProxy(DATA_DEVICE)

for proxy in (scan, microscope, data):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())
print("data      :", data.state())


## 2. Configure the DATA Tango device

In [ ]:
data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = SAVE_PATH


print(json.dumps(json.loads(data.get_config()), indent=2))


In [ ]:
# If ThermoMicroscope was registered with data_device_address, this should reflect the DATA device settings.
# If it does not, update the Tango DB property data_device_address = asyncroscopy/data/default and restart ThermoMicroscope.
try:
    print(json.dumps(json.loads(microscope.get_tiled_acquisition_config()), indent=2))
except Exception as exc:
    print("Could not read microscope data config:", exc)


## 3. Configure a small STEM acquisition

In [ ]:
scan.Activate(["haadf"])
scan.dwell_time = 1e-6
scan.imsize = 128

print("dwell_time:", scan.dwell_time)
print("imsize    :", scan.imsize)


## 4. Acquire and inspect the saved path

In [ ]:
saved_path = microscope.get_scanned_image()

print("Saved file:", saved_path)

print("\nDATA device path check:")
try:
    print(json.dumps(json.loads(data.path_exists(saved_path)), indent=2))
except Exception as exc:
    print("Could not run data.path_exists:", exc)


## 5. Check Tiled root

In [ ]:
print("Tiled root entries:")
try:
    print(json.dumps(json.loads(data.list_root()), indent=2)[:4000])
except Exception as exc:
    print("Could not list Tiled root:", exc)


## 6. Resolve the saved path through Tiled

In [ ]:
def get_data_with_retry(saved_path, timeout=30, interval=2):
    start = time.time()
    last_error = None
    while time.time() - start < timeout:
        try:
            return json.loads(data.get_data(saved_path))
        except Exception as exc:
            last_error = exc
            print("Tiled lookup not ready yet:", exc)
            time.sleep(interval)

    raise RuntimeError(f"Could not resolve data through Tiled after {timeout}s") from last_error

data = get_data_with_retry(saved_path)

if isinstance(data, dict):
    summary = {k: data[k] for k in data.keys() if k != "data"}
    print(json.dumps(summary, indent=2)[:4000])
else:
    print(type(data), str(data)[:1000])


## 7. Plot if Tiled returned an array

In [ ]:
if isinstance(data, dict) and data.get("type") == "ndarray":
    image = np.asarray(data["data"], dtype=np.dtype(data["dtype"]))
    image = image.reshape(data["shape"])

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image, cmap="gray", interpolation="none")
    ax.set_title("HAADF from Tiled")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Tiled did not return a direct ndarray. Inspect data and traverse the returned node/path as needed.")
